# NoisePQC++ Benchmark Analysis

This notebook analyzes handshake performance across Classical, Post-Quantum (PQ), and Hybrid Forward Secrecy (HFS) modes.

## Prerequisites
1. Run `pip install -r requirements.txt`
2. Generate benchmark data by running:
   ```bash
   ./bin/Release/benchmarks/bench_handshake --benchmark_format=json --benchmark_out=analysis/data/handshake_results.json
   ```
3. Convert to CSV:
   ```bash
   python scripts/json_to_csv.py --split data/handshake_results.json data/
   ```

In [ ]:
# Cell 1: Imports and Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Optional

# Publication-quality settings
plt.rcParams.update({
    'font.size': 12,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'figure.figsize': (10, 6),
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Colorblind-friendly palette (Wong palette)
COLORS = {
    'Classical': '#0072B2',    # Blue
    'PQ': '#D55E00',           # Vermillion/Orange
    'HFS': '#009E73',          # Green
    'X25519': '#56B4E9',       # Sky blue
    'X448': '#E69F00',         # Orange/Yellow
    'M512': '#CC79A7',         # Pink
    'M768': '#F0E442',         # Yellow
    'M1024': '#999999',        # Gray
}

# Paths
DATA_DIR = Path('data/')
OUTPUT_DIR = Path('figures/')
OUTPUT_DIR.mkdir(exist_ok=True)

print("Configuration loaded.")
print(f"Data directory: {DATA_DIR.absolute()}")
print(f"Output directory: {OUTPUT_DIR.absolute()}")

In [ ]:
# Cell 2: Load Benchmark Data

def load_csv_if_exists(path: Path) -> Optional[pd.DataFrame]:
    """Load CSV if it exists, return None otherwise."""
    if path.exists():
        df = pd.read_csv(path)
        print(f"Loaded {len(df)} rows from {path.name}")
        return df
    else:
        print(f"Warning: {path} not found")
        return None

# Load local benchmark CSVs
df_classical = load_csv_if_exists(DATA_DIR / 'handshake_classical.csv')
df_pq = load_csv_if_exists(DATA_DIR / 'handshake_pq.csv')
df_hfs = load_csv_if_exists(DATA_DIR / 'handshake_hfs.csv')
df_keygen = load_csv_if_exists(DATA_DIR / 'handshake_keygen.csv')

# Load network benchmark CSVs (if available)
df_network = load_csv_if_exists(DATA_DIR / 'network_results.csv')
df_stress = load_csv_if_exists(DATA_DIR / 'stress_results.csv')

In [ ]:
# Cell 3: Data Preview

print("=" * 60)
print("Classical Handshake Data")
print("=" * 60)
if df_classical is not None:
    display(df_classical.head())
    print(f"\nPatterns: {df_classical['pattern'].unique()}")
    print(f"Algorithms: {df_classical['algo'].unique()}")

print("\n" + "=" * 60)
print("Post-Quantum Handshake Data")
print("=" * 60)
if df_pq is not None:
    display(df_pq.head())
    print(f"\nPatterns: {df_pq['pattern'].unique()}")
    print(f"Algorithms: {df_pq['algo'].unique()}")

In [ ]:
# Cell 4: Latency Comparison Bar Chart

def plot_latency_comparison(df_c, df_pq, df_hfs, save=True):
    """Create grouped bar chart comparing latency across modes."""
    if df_c is None and df_pq is None and df_hfs is None:
        print("No data available for latency comparison")
        return
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Define base patterns to compare
    base_patterns = ['NN', 'XX', 'IK', 'KK']
    
    data = []
    
    # Classical (X25519 only for clean comparison)
    if df_c is not None:
        for pattern in base_patterns:
            mask = (df_c['pattern'] == pattern) & (df_c['algo'] == 'X25519')
            if mask.any():
                latency = df_c.loc[mask, 'real_time_us'].mean()
                data.append({'Pattern': pattern, 'Mode': 'Classical (X25519)', 'Latency': latency})
    
    # Post-Quantum (M768)
    if df_pq is not None:
        for pattern in base_patterns:
            pq_pattern = f'pq{pattern}'
            mask = (df_pq['pattern'] == pq_pattern) & (df_pq['algo'] == 'M768')
            if mask.any():
                latency = df_pq.loc[mask, 'real_time_us'].mean()
                data.append({'Pattern': pattern, 'Mode': 'PQ (ML-KEM-768)', 'Latency': latency})
    
    # HFS (X25519+M768)
    if df_hfs is not None:
        for pattern in base_patterns:
            hfs_pattern = f'{pattern}hfs'
            mask = (df_hfs['pattern'] == hfs_pattern) & (df_hfs['algo'].str.contains('X25519') if 'algo' in df_hfs.columns else True)
            if mask.any():
                latency = df_hfs.loc[mask, 'real_time_us'].mean()
                data.append({'Pattern': pattern, 'Mode': 'HFS (X25519+M768)', 'Latency': latency})
    
    if not data:
        print("No matching data found for comparison")
        return
    
    df_plot = pd.DataFrame(data)
    
    # Create grouped bar chart
    patterns = df_plot['Pattern'].unique()
    modes = df_plot['Mode'].unique()
    x = np.arange(len(patterns))
    width = 0.25
    
    for i, mode in enumerate(modes):
        mode_data = df_plot[df_plot['Mode'] == mode]
        latencies = [mode_data[mode_data['Pattern'] == p]['Latency'].values[0] 
                     if len(mode_data[mode_data['Pattern'] == p]) > 0 else 0 
                     for p in patterns]
        
        color = COLORS['Classical'] if 'Classical' in mode else \
                COLORS['PQ'] if 'PQ' in mode else COLORS['HFS']
        
        bars = ax.bar(x + i*width, latencies, width, label=mode, color=color)
        
        # Add value labels on bars
        for bar, val in zip(bars, latencies):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                       f'{val:.0f}', ha='center', va='bottom', fontsize=8)
    
    ax.set_xlabel('Handshake Pattern')
    ax.set_ylabel('Latency (microseconds)')
    ax.set_title('Handshake Latency: Classical vs Post-Quantum vs Hybrid')
    ax.set_xticks(x + width)
    ax.set_xticklabels(patterns)
    ax.legend()
    
    plt.tight_layout()
    
    if save:
        plt.savefig(OUTPUT_DIR / 'fig_latency_comparison.png', bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'fig_latency_comparison.pdf', bbox_inches='tight')
        print(f"Saved to {OUTPUT_DIR / 'fig_latency_comparison.png'}")
    
    plt.show()

plot_latency_comparison(df_classical, df_pq, df_hfs)

In [ ]:
# Cell 5: CPU Time Analysis

def plot_cpu_time(df_c, df_pq, df_hfs, save=True):
    """Plot CPU time by pattern and mode."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    datasets = [
        (df_c, 'Classical', COLORS['Classical']),
        (df_pq, 'Post-Quantum', COLORS['PQ']),
        (df_hfs, 'Hybrid (HFS)', COLORS['HFS']),
    ]
    
    for ax, (df, title, color) in zip(axes, datasets):
        if df is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title)
            continue
            
        patterns = df['pattern'].unique()
        cpu_times = [df[df['pattern'] == p]['cpu_time_us'].mean() for p in patterns]
        
        bars = ax.bar(patterns, cpu_times, color=color)
        ax.set_xlabel('Pattern')
        ax.set_ylabel('CPU Time (microseconds)')
        ax.set_title(f'{title} Handshakes')
        ax.tick_params(axis='x', rotation=45)
        
        # Add value labels
        for bar, val in zip(bars, cpu_times):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                   f'{val:.0f}', ha='center', va='bottom', fontsize=8)
    
    plt.tight_layout()
    
    if save:
        plt.savefig(OUTPUT_DIR / 'fig_cpu_time.png', bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'fig_cpu_time.pdf', bbox_inches='tight')
        print(f"Saved to {OUTPUT_DIR / 'fig_cpu_time.png'}")
    
    plt.show()

plot_cpu_time(df_classical, df_pq, df_hfs)

In [ ]:
# Cell 6: Key Size Comparison

def plot_key_sizes(save=True):
    """Plot public key and ciphertext sizes for different algorithms."""
    # Known key sizes (from Noise/ML-KEM specifications)
    key_sizes = {
        'X25519': {'public_key': 32, 'shared_secret': 32, 'ciphertext': 0},
        'X448': {'public_key': 56, 'shared_secret': 56, 'ciphertext': 0},
        'ML-KEM-512': {'public_key': 800, 'shared_secret': 32, 'ciphertext': 768},
        'ML-KEM-768': {'public_key': 1184, 'shared_secret': 32, 'ciphertext': 1088},
        'ML-KEM-1024': {'public_key': 1568, 'shared_secret': 32, 'ciphertext': 1568},
    }
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    algos = list(key_sizes.keys())
    x = np.arange(len(algos))
    width = 0.35
    
    # Public key sizes
    pub_keys = [key_sizes[a]['public_key'] for a in algos]
    colors = [COLORS['X25519'] if 'X25519' in a else 
              COLORS['X448'] if 'X448' in a else
              COLORS['M512'] if '512' in a else
              COLORS['M768'] if '768' in a else
              COLORS['M1024'] for a in algos]
    
    bars1 = ax1.bar(x, pub_keys, color=colors)
    ax1.set_xlabel('Algorithm')
    ax1.set_ylabel('Size (bytes)')
    ax1.set_title('Public Key Sizes')
    ax1.set_xticks(x)
    ax1.set_xticklabels(algos, rotation=45, ha='right')
    
    for bar, val in zip(bars1, pub_keys):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val}', ha='center', va='bottom', fontsize=9)
    
    # Ciphertext/DH sizes (for KEMs)
    ciphertexts = [key_sizes[a]['ciphertext'] if key_sizes[a]['ciphertext'] > 0 
                   else key_sizes[a]['public_key'] for a in algos]
    
    bars2 = ax2.bar(x, ciphertexts, color=colors)
    ax2.set_xlabel('Algorithm')
    ax2.set_ylabel('Size (bytes)')
    ax2.set_title('Ciphertext / DH Exchange Sizes')
    ax2.set_xticks(x)
    ax2.set_xticklabels(algos, rotation=45, ha='right')
    
    for bar, val in zip(bars2, ciphertexts):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    if save:
        plt.savefig(OUTPUT_DIR / 'fig_key_sizes.png', bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'fig_key_sizes.pdf', bbox_inches='tight')
        print(f"Saved to {OUTPUT_DIR / 'fig_key_sizes.png'}")
    
    plt.show()

plot_key_sizes()

In [ ]:
# Cell 7: Message Overhead Analysis

def plot_message_overhead(df_c, df_pq, df_hfs, save=True):
    """Plot bytes per handshake across modes."""
    if df_c is None and df_pq is None and df_hfs is None:
        print("No data available")
        return
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    base_patterns = ['NN', 'XX', 'IK', 'KK']
    data = []
    
    # Classical
    if df_c is not None and 'bytes_per_hs' in df_c.columns:
        for pattern in base_patterns:
            mask = (df_c['pattern'] == pattern) & (df_c['algo'] == 'X25519')
            if mask.any():
                bytes_hs = df_c.loc[mask, 'bytes_per_hs'].mean()
                data.append({'Pattern': pattern, 'Mode': 'Classical', 'Bytes': bytes_hs})
    
    # PQ
    if df_pq is not None and 'bytes_per_hs' in df_pq.columns:
        for pattern in base_patterns:
            pq_pattern = f'pq{pattern}'
            mask = (df_pq['pattern'] == pq_pattern) & (df_pq['algo'] == 'M768')
            if mask.any():
                bytes_hs = df_pq.loc[mask, 'bytes_per_hs'].mean()
                data.append({'Pattern': pattern, 'Mode': 'PQ', 'Bytes': bytes_hs})
    
    # HFS
    if df_hfs is not None and 'bytes_per_hs' in df_hfs.columns:
        for pattern in base_patterns:
            hfs_pattern = f'{pattern}hfs'
            mask = df_hfs['pattern'] == hfs_pattern
            if mask.any():
                bytes_hs = df_hfs.loc[mask, 'bytes_per_hs'].mean()
                data.append({'Pattern': pattern, 'Mode': 'HFS', 'Bytes': bytes_hs})
    
    if not data:
        print("No bytes_per_hs data available")
        return
    
    df_plot = pd.DataFrame(data)
    
    # Grouped bar chart
    patterns = df_plot['Pattern'].unique()
    modes = df_plot['Mode'].unique()
    x = np.arange(len(patterns))
    width = 0.25
    
    for i, mode in enumerate(modes):
        mode_data = df_plot[df_plot['Mode'] == mode]
        bytes_vals = [mode_data[mode_data['Pattern'] == p]['Bytes'].values[0] 
                      if len(mode_data[mode_data['Pattern'] == p]) > 0 else 0 
                      for p in patterns]
        
        color = COLORS.get(mode, '#888888')
        bars = ax.bar(x + i*width, bytes_vals, width, label=mode, color=color)
        
        for bar, val in zip(bars, bytes_vals):
            if val > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                       f'{val:.0f}', ha='center', va='bottom', fontsize=8)
    
    ax.set_xlabel('Handshake Pattern')
    ax.set_ylabel('Total Bytes Exchanged')
    ax.set_title('Handshake Message Overhead: Bytes per Handshake')
    ax.set_xticks(x + width)
    ax.set_xticklabels(patterns)
    ax.legend()
    
    plt.tight_layout()
    
    if save:
        plt.savefig(OUTPUT_DIR / 'fig_message_overhead.png', bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'fig_message_overhead.pdf', bbox_inches='tight')
        print(f"Saved to {OUTPUT_DIR / 'fig_message_overhead.png'}")
    
    plt.show()

plot_message_overhead(df_classical, df_pq, df_hfs)

In [ ]:
# Cell 8: Network RTT vs Artificial Latency (if network data available)

def plot_network_rtt(df_net, save=True):
    """Plot handshake RTT vs artificial network latency."""
    if df_net is None:
        print("No network data available")
        print("Run network benchmarks and provide network_results.csv")
        return
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    modes = df_net['mode'].unique()
    
    for mode in modes:
        mode_data = df_net[df_net['mode'] == mode]
        
        # Group by latency and get mean RTT
        if 'latency_ms' in mode_data.columns and 'rtt_ns' in mode_data.columns:
            grouped = mode_data.groupby('latency_ms')['rtt_ns'].mean() / 1e6  # ns to ms
            
            color = COLORS['Classical'] if mode in ['nn', 'xx', 'ik', 'kk'] else \
                    COLORS['PQ'] if mode.startswith('pq') else COLORS['HFS']
            
            ax.plot(grouped.index, grouped.values, 'o-', label=mode.upper(), color=color, markersize=8)
    
    ax.set_xlabel('Artificial Network Latency (ms)')
    ax.set_ylabel('Total Handshake RTT (ms)')
    ax.set_title('Network Handshake RTT vs Artificial Latency')
    ax.legend()
    
    plt.tight_layout()
    
    if save:
        plt.savefig(OUTPUT_DIR / 'fig_network_rtt.png', bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'fig_network_rtt.pdf', bbox_inches='tight')
        print(f"Saved to {OUTPUT_DIR / 'fig_network_rtt.png'}")
    
    plt.show()

plot_network_rtt(df_network)

In [ ]:
# Cell 9: Throughput Scaling (if stress data available)

def plot_throughput(df_stress, save=True):
    """Plot throughput vs concurrent clients."""
    if df_stress is None:
        print("No stress test data available")
        print("Run stress benchmarks and provide stress_results.csv")
        return
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    modes = df_stress['mode'].unique()
    
    for mode in modes:
        mode_data = df_stress[df_stress['mode'] == mode]
        
        if 'clients' in mode_data.columns and 'throughput' in mode_data.columns:
            clients = mode_data['clients'].values
            throughput = mode_data['throughput'].values
            
            color = COLORS['Classical'] if mode in ['nn', 'xx', 'ik', 'kk'] else \
                    COLORS['PQ'] if mode.startswith('pq') else COLORS['HFS']
            
            ax.plot(clients, throughput, 'o-', label=mode.upper(), color=color, markersize=8)
    
    ax.set_xlabel('Number of Concurrent Clients')
    ax.set_ylabel('Handshakes per Second')
    ax.set_title('Throughput Scaling with Concurrent Clients')
    ax.legend()
    
    plt.tight_layout()
    
    if save:
        plt.savefig(OUTPUT_DIR / 'fig_throughput_scaling.png', bbox_inches='tight')
        plt.savefig(OUTPUT_DIR / 'fig_throughput_scaling.pdf', bbox_inches='tight')
        print(f"Saved to {OUTPUT_DIR / 'fig_throughput_scaling.png'}")
    
    plt.show()

plot_throughput(df_stress)

In [ ]:
# Cell 10: Summary Statistics Table

def generate_summary_table(df_c, df_pq, df_hfs):
    """Generate summary statistics table."""
    summary = []
    
    datasets = [
        ('Classical', df_c),
        ('Post-Quantum', df_pq),
        ('Hybrid (HFS)', df_hfs),
    ]
    
    for mode, df in datasets:
        if df is None:
            continue
            
        for pattern in df['pattern'].unique():
            pattern_data = df[df['pattern'] == pattern]
            
            row = {
                'Mode': mode,
                'Pattern': pattern,
                'Algorithm': pattern_data['algo'].iloc[0] if 'algo' in pattern_data.columns else 'N/A',
                'Mean Latency (us)': pattern_data['real_time_us'].mean(),
                'CPU Time (us)': pattern_data['cpu_time_us'].mean(),
            }
            
            if 'bytes_per_hs' in pattern_data.columns:
                row['Bytes/Handshake'] = pattern_data['bytes_per_hs'].mean()
            
            if 'messages' in pattern_data.columns:
                row['Messages'] = pattern_data['messages'].iloc[0]
            
            summary.append(row)
    
    if not summary:
        print("No data for summary table")
        return None
    
    df_summary = pd.DataFrame(summary)
    
    # Format numbers
    format_dict = {
        'Mean Latency (us)': '{:.1f}',
        'CPU Time (us)': '{:.1f}',
        'Bytes/Handshake': '{:.0f}',
    }
    
    return df_summary.style.format(format_dict).background_gradient(
        subset=['Mean Latency (us)'], cmap='YlOrRd'
    )

summary_table = generate_summary_table(df_classical, df_pq, df_hfs)
if summary_table is not None:
    display(summary_table)

In [ ]:
# Cell 11: Export Summary to CSV

def export_summary(df_c, df_pq, df_hfs):
    """Export summary data to CSV."""
    summary = []
    
    datasets = [
        ('Classical', df_c),
        ('Post-Quantum', df_pq),
        ('Hybrid (HFS)', df_hfs),
    ]
    
    for mode, df in datasets:
        if df is None:
            continue
            
        for pattern in df['pattern'].unique():
            pattern_data = df[df['pattern'] == pattern]
            
            row = {
                'mode': mode,
                'pattern': pattern,
                'algorithm': pattern_data['algo'].iloc[0] if 'algo' in pattern_data.columns else '',
                'mean_latency_us': pattern_data['real_time_us'].mean(),
                'cpu_time_us': pattern_data['cpu_time_us'].mean(),
                'bytes_per_hs': pattern_data['bytes_per_hs'].mean() if 'bytes_per_hs' in pattern_data.columns else 0,
                'messages': pattern_data['messages'].iloc[0] if 'messages' in pattern_data.columns else 0,
            }
            summary.append(row)
    
    if summary:
        df_summary = pd.DataFrame(summary)
        output_path = OUTPUT_DIR / 'summary_stats.csv'
        df_summary.to_csv(output_path, index=False)
        print(f"Summary exported to {output_path}")
        return df_summary
    
    return None

export_summary(df_classical, df_pq, df_hfs)

## Analysis Complete

All figures have been saved to the `figures/` directory:
- `fig_latency_comparison.png/pdf` - Latency comparison across modes
- `fig_cpu_time.png/pdf` - CPU time by pattern
- `fig_key_sizes.png/pdf` - Public key and ciphertext sizes
- `fig_message_overhead.png/pdf` - Bytes per handshake
- `fig_network_rtt.png/pdf` - Network RTT (if data available)
- `fig_throughput_scaling.png/pdf` - Throughput scaling (if data available)
- `summary_stats.csv` - Summary statistics table